# Laboratorio 1 · Parte 2: Extracción y carga

Aquí construyes el primer tramo del pipeline: sacar los tweets de la API de Hugging Face y dejarlos en el volumen `raw` de Databricks, sin cambiarlos.

```
API de Hugging Face ──extraer──▶ JSON ──aplanar──▶ tabla ──guardar──▶ Parquet ──cargar──▶ /Volumes/workspace/dlbd/raw/tweets/
```

**Dataset:** `cardiffnlp/tweet_sentiment_multilingual`, configuración `spanish`. Tiene 3 030 tweets en tres splits (`train`, `validation`, `test`) y una etiqueta de sentimiento: 0 negativo, 1 neutral, 2 positivo.

La API de Hugging Face es pública: no necesita cuenta ni token. Para Databricks necesitas los mismos secretos de la Parte 1: `DATABRICKS_HOST` y `DATABRICKS_TOKEN`. Completa las celdas `# TODO`; las celdas de verificación te dicen si vas bien.

## Preparación

In [ ]:
import os
import time

import pandas as pd
import requests
from google.colab import userdata

DATASET = "cardiffnlp/tweet_sentiment_multilingual"
CONFIG = "spanish"
SPLITS = ["train", "validation", "test"]
ESPERADO = {"train": 1838, "validation": 323, "test": 869}  # filas por split

API_FILAS = "https://datasets-server.huggingface.co/rows"
PAGINA = 100  # la API entrega como máximo 100 filas por llamada

## 1. Extraer

La API de filas de Hugging Face recibe estos parámetros:

| Parámetro | Ejemplo | Significado |
|---|---|---|
| `dataset` | `cardiffnlp/tweet_sentiment_multilingual` | el dataset |
| `config` | `spanish` | la configuración (idioma) |
| `split` | `train` | el split |
| `offset` | `0`, `100`, `200`… | desde qué fila empezar |
| `length` | `100` | cuántas filas traer (máximo 100) |

Como solo entrega 100 filas por llamada, hay que **paginar**: pedir la página desde 0, luego desde 100, y así hasta llegar a `num_rows_total`.

La función `pedir_pagina` ya está hecha. Ejecuta la celda de exploración para ver cómo responde la API.

In [ ]:
def pedir_pagina(split, offset, length=PAGINA):
    """Pide una página de filas a la API. Reintenta si la API está saturada."""
    params = {"dataset": DATASET, "config": CONFIG, "split": split,
              "offset": offset, "length": length}
    for intento in range(4):
        r = requests.get(API_FILAS, params=params, timeout=60)
        if r.status_code in (429, 500, 502, 503):
            time.sleep(2 ** intento)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

In [ ]:
# Exploración: una página pequeña de train
muestra = pedir_pagina("train", 0, length=2)
print("Llaves de la respuesta:", list(muestra.keys()))
print("Filas totales en train:", muestra["num_rows_total"])
muestra["rows"]

In [ ]:
def extraer_split(split):
    filas = []
    offset = 0
    total = None
    while total is None or offset < total:
        pagina = pedir_pagina(split, offset)
        total = pagina["num_rows_total"]
        filas.extend(pagina["rows"])
        offset += PAGINA
    print(f"{split}: {len(filas)} filas")
    return filas

In [ ]:
crudo = {split: extraer_split(split) for split in SPLITS}

for split, n in ESPERADO.items():
    assert len(crudo[split]) == n, f"{split}: se esperaban {n} filas y llegaron {len(crudo[split])}"
print(f"✅ Extracción completa: {sum(len(f) for f in crudo.values())} filas")

## 2. Aplanar

Cada fila de la API viene envuelta así:

```json
{"row_idx": 0, "row": {"text": "...", "label": 2}, "truncated_cells": []}
```

Lo que nos interesa está dentro de `row`. Aplanar es convertir esa lista de diccionarios en una tabla (DataFrame) con tres columnas: `text`, `label` y `split`. **No se modifica el texto.**

In [ ]:
def aplanar(filas, split):
    df = pd.DataFrame([f["row"] for f in filas])
    df["split"] = split
    return df[["text", "label", "split"]]

In [ ]:
tablas = {split: aplanar(filas, split) for split, filas in crudo.items()}

for split, df in tablas.items():
    assert list(df.columns) == ["text", "label", "split"], f"{split}: columnas incorrectas {list(df.columns)}"
    assert len(df) == ESPERADO[split], f"{split}: cantidad de filas incorrecta"
    assert set(df["label"]) <= {0, 1, 2}, f"{split}: hay etiquetas fuera de 0, 1, 2"
assert all(not f["truncated_cells"] for filas in crudo.values() for f in filas), "La API truncó algún texto"

print("✅ Tablas listas")
tablas["train"].head()

## 3. Guardar

Guarda **un archivo Parquet por split** en la carpeta local `datos/`, con el nombre `tweets_<split>.parquet`.

Parquet es un formato columnar y comprimido: ocupa menos que un CSV y conserva los tipos de cada columna.

In [ ]:
os.makedirs("datos", exist_ok=True)

for split, df in tablas.items():
    df.to_parquet(f"datos/tweets_{split}.parquet", index=False)

In [ ]:
for split, df in tablas.items():
    leido = pd.read_parquet(f"datos/tweets_{split}.parquet")
    assert leido.equals(df.reset_index(drop=True)), f"{split}: el Parquet no coincide con la tabla"
    print(f"datos/tweets_{split}.parquet  {os.path.getsize(f'datos/tweets_{split}.parquet'):>8,} bytes")
print("✅ Archivos guardados")

## 4. Cargar

Sube los tres archivos al volumen de Databricks con la Files API. Para subir un archivo se hace un `PUT` con el contenido del archivo en el cuerpo:

```
PUT {DATABRICKS_HOST}/api/2.0/fs/files{ruta_remota}?overwrite=true
```

La carpeta de destino es `/Volumes/workspace/dlbd/raw/tweets/`. La celda siguiente ya la crea.

In [ ]:
from urllib.parse import urlparse

# Se queda solo con https://<servidor>, aunque el secreto traiga la URL completa del navegador (/?o=..., /browse, etc.)
host = userdata.get("DATABRICKS_HOST").strip()
DATABRICKS_HOST = "https://" + urlparse(host if host.startswith("http") else "https://" + host).netloc
print("Servidor de Databricks:", DATABRICKS_HOST)
DB_HEADERS = {"Authorization": f"Bearer {userdata.get('DATABRICKS_TOKEN').strip()}"}

DESTINO = "/Volumes/workspace/dlbd/raw/tweets"
requests.put(f"{DATABRICKS_HOST}/api/2.0/fs/directories{DESTINO}", headers=DB_HEADERS, timeout=30).raise_for_status()

In [ ]:
def subir(ruta_local, ruta_remota):
    with open(ruta_local, "rb") as f:
        r = requests.put(f"{DATABRICKS_HOST}/api/2.0/fs/files{ruta_remota}",
                         headers=DB_HEADERS, params={"overwrite": "true"},
                         data=f, timeout=120)
    r.raise_for_status()

for split in SPLITS:
    subir(f"datos/tweets_{split}.parquet", f"{DESTINO}/tweets_{split}.parquet")

In [ ]:
r = requests.get(f"{DATABRICKS_HOST}/api/2.0/fs/directories{DESTINO}", headers=DB_HEADERS, timeout=30)
r.raise_for_status()
en_volumen = sorted(c["name"] for c in (r.json() if r.text.strip() else {}).get("contents", []))
print(en_volumen)

assert en_volumen == sorted(f"tweets_{s}.parquet" for s in SPLITS), "Faltan archivos en el volumen"
print("✅ Carga completa")

## Entrega

Guarda este notebook **con sus salidas** en tu repositorio: **Archivo → Guardar una copia en GitHub**, ruta `lab1/parte2_extraccion.ipynb`.

Sigue la Parte 3 en Databricks: `lab1/parte3_bronce.sql`.